In [9]:
import sys
sys.path.append("..")  # pour importer le dossier sources/, situé à la racine du dépôt

In [ ]:
# Chargement des 8 sessions : téléchargées une seule fois, puis relues depuis data/raw/
import pandas as pd

from sources.chargement_data import charger_parcoursup, charger_poursuite, charger_ips, DATASETS
from sources.cleaning_data import preparer_base, harmoniser_ips, preparer_poursuite, base_selective
from sources.jointures import joindre_ips, panel_academies
from sources.indicateurs import (taux_poursuite, part_poursuite_autre_academie_idf,
                                 part_admis_meme_academie, boursiers_admis_et_candidats)

dfs = {annee: charger_parcoursup(annee) for annee in DATASETS}

Préparation de la base : identifiant de formation suivi dans le temps (`code`), sélectivité (`selective`) et filière (`filiere`) harmonisées, puis empilement des sessions. Les choix sont justifiés dans `notebooks/exploration.ipynb`.

In [11]:
base = preparer_base(dfs)

print(base.shape)
print("sélectivité manquante :", base["selective"].isna().sum())
base.groupby("session").size()

(104274, 133)
sélectivité manquante : 634


session
2018    10697
2019    11577
2020    12760
2021    13396
2022    13644
2023    13869
2024    14079
2025    14252
dtype: int64

## Enrichissement 1 : l'indice de position sociale (IPS) des lycées

L'IPS, publié par l'Éducation nationale, résume le milieu social des élèves d'un lycée : plus il est élevé, plus le milieu est favorisé. Beaucoup de CPGE et de BTS sont installés dans des lycées. On peut donc rattacher à chaque formation l'IPS du lycée qui l'accueille, grâce au code UAI et à l'année. Les trois fichiers publiés (2016-2021, 2022, 2023-2025) n'ont pas les mêmes formats : on les harmonise avant la jointure.

In [12]:
ips = harmoniser_ips(charger_ips())
base = joindre_ips(base, ips)

# Part des formations qui trouvent l'IPS de leur lycée, par filière (en %)
avec_ips = base["ips_lycee"].notna()
(avec_ips.groupby(base["filiere"]).mean() * 100).round(1).sort_values(ascending=False)

filiere
CPGE                   94.4
BTS                    85.5
Autre                  56.1
Ecole d'ingénieur       4.6
Sanitaire et social     1.0
IUT                     0.1
Ecole de commerce       0.0
LAS                     0.0
Licence                 0.0
Santé                   0.0
Name: ips_lycee, dtype: float64

94 % des CPGE et 86 % des BTS trouvent l'IPS de leur lycée. Les universités, les IUT et les écoles ne sont pas des lycées : la jointure ne les concerne pas. L'analyse fondée sur l'IPS portera donc sur les CPGE et les BTS.

## Enrichissement 2 : le taux de poursuite des néo-bacheliers dans le supérieur (2010-2025)

Ce fichier du ministère donne, pour chaque académie où le bac a été obtenu, chaque année et chaque voie du bac, le nombre de bacheliers et le nombre de ceux qui poursuivent dans le supérieur. Il commence en 2010 : on peut comparer la période APB (jusqu'en 2017) à la période Parcoursup (depuis 2018).

In [13]:
poursuite = preparer_poursuite(charger_poursuite())

# Taux de poursuite (en %) par année et par voie du bac : une colonne par voie
taux_poursuite(poursuite, ["annee", "voie"]).unstack()

voie,Bac général,Bac professionnel,Bac technologique
annee,,,
2010,94.5,34.8,80.5
2011,93.6,34.5,80.5
2012,93.4,35.1,80.4
2013,93.2,39.5,77.2
2014,93.3,35.7,78.8
2015,93.3,36.5,80.7
2016,92.3,36.1,79.4
2017,91.8,38.6,79.0
2018,92.8,39.5,78.7


Le taux de poursuite des bacheliers professionnels passe de 34,8 % en 2010 à 38,6 % en 2017, puis à 46,5 % en 2025. Celui des bacheliers généraux (environ 93 %) et technologiques (environ 80 %) reste stable.

On relie ensuite ce fichier aux admis de Parcoursup, par académie et par année. On obtient un tableau avec une ligne par académie et par année : c'est la jointure entre les deux sources. Attention : Parcoursup donne l'académie de la formation, et le fichier de poursuite l'académie du bac.

In [14]:
panel = panel_academies(base, poursuite)

print(panel.shape, "-", panel["academie"].nunique(), "académies")
panel.head()

(240, 13) - 30 académies


,academie,annee,acc_tot,acc_neobac,acc_bp,acc_bt,acc_brs,bacheliers_Bac général,bacheliers_Bac professionnel,bacheliers_Bac technologique,poursuivants_Bac général,poursuivants_Bac professionnel,poursuivants_Bac technologique
0,Aix-Marseille,2018,24485,18607,2068,3850,4363,15646,7291,6172,14346.33570,2568.58960,4569.22180
1,Aix-Marseille,2019,27080,19253,2287,4081,4614,15268,7063,6176,14116.18750,2891.03480,4783.35260
2,Aix-Marseille,2020,28793,21222,2484,4331,6454,17001,7636,6767,15693.57670,3437.52100,4947.35940
3,Aix-Marseille,2021,27614,19604,2143,3542,5888,16450,7179,5887,15659.18680,3515.97980,4461.22740
4,Aix-Marseille,2022,26776,18561,1911,3455,5708,16159,6812,5649,15171.71334,3293.76365,4450.19656


## Zoom Île-de-France : le secteur unique de 2019

Depuis la session 2019, les académies de Paris, Créteil et Versailles forment un seul secteur dans Parcoursup. On regarde l'effet de ce changement dans les deux sources :
- Parcoursup : la part des admis venant de l'académie de Paris, dans les formations parisiennes ;
- poursuite d'études : la part des bacheliers franciliens qui poursuivent dans une autre académie d'Île-de-France.

In [ ]:
# Les deux indicateurs côte à côte, en %, de 2016 à 2025 (Parcoursup commence en 2018)
idf = pd.DataFrame({
    "admis de l'académie de Paris dans les formations parisiennes": part_admis_meme_academie(base, "Paris"),
    "bacheliers franciliens poursuivant dans une autre académie d'IDF": part_poursuite_autre_academie_idf(poursuite),
})
idf.loc[2016:]

## Formations sélectives et non sélectives : la place des boursiers

La sélection est l'étape où Parcoursup peut favoriser ou freiner la démocratisation : ce sont les formations sélectives qui choisissent leurs admis. On les compare aux formations non sélectives (surtout les licences), qui accueillent une grande partie des boursiers et des bacheliers professionnels.

Les formations sélectives représentent 79 % des formations de la base.

In [ ]:
formations_selectives = base_selective(base)

print(len(formations_selectives), "formations sélectives sur", len(base))

Pour mesurer la place des boursiers, on ne compare pas la part de boursiers d'une année à l'autre : le nombre de boursiers recensés fait un saut en 2020 et chute en 2025, aussi bien parmi les vœux que parmi les admis. Ces sauts viennent du comptage, pas de la sélection.

On compare donc, la même année, la part de boursiers parmi les vœux des néo-bacheliers et parmi les néo-bacheliers admis. Le rapport des deux vaut 1 si les boursiers sont aussi présents parmi les admis que parmi les vœux ; au-dessus de 1, ils sont plus présents parmi les admis.

In [ ]:
boursiers = boursiers_admis_et_candidats(base, ["session", "selective"])

# Rapport admis / vœux : une colonne pour les non sélectives (0) et une pour les sélectives (1)
boursiers["rapport"].unstack()

Chaque année, le rapport est supérieur à 1 : les boursiers sont un peu plus présents parmi les admis que parmi les vœux. L'écart est plus faible dans les formations sélectives (entre 1,04 et 1,11 depuis 2019) que dans les non sélectives (entre 1,15 et 1,23). Il ne s'élargit pas depuis 2019. En 2018, la sélectivité est reportée depuis 2020 : la valeur de cette année est à lire avec prudence.

Limites : ce sont des vœux et non des candidats (un candidat fait plusieurs vœux), et un rapport supérieur à 1 peut aussi venir des choix des candidats (vœux plus nombreux chez les non-boursiers, par exemple). On regarde ensuite le détail par filière parmi les formations sélectives.

In [ ]:
# Même rapport parmi les formations sélectives, pour les principales filières
principales = formations_selectives[formations_selectives["filiere"].isin(["CPGE", "BTS", "IUT"])]
boursiers_admis_et_candidats(principales, ["session", "filiere"])["rapport"].unstack()

En BTS, le rapport reste proche de 1 chaque année : les boursiers sont admis dans la même proportion que leur part dans les vœux. En CPGE, il est plus élevé (entre 1,10 et 1,21) : parmi les néo-bacheliers admis en CPGE, les boursiers sont un peu plus présents que parmi les vœux. Ce sont des descriptions : elles ne disent pas si cet écart vient de la sélection des formations (par exemple des quotas de boursiers) ou des choix des candidats.